# Agent + MCP demo: feature retrieval and drift detection

This executed demo calls both platform .CP servers through their streamable HTTP transport. It keeps the specialist requests explicit and bounded: feature/RAG context for one ticker, then a deterministic drift report for the same ticker.

In [1]:
import asyncio
import json
import os

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

FEATURE_MCP_URL = os.environ.get("FEATURE_MCP_URL", "http://feature-mcp.dataflow.svc.cluster.local/mcp")
DRIFT_MCP_URL = os.environ.get("DRIFT_MCP_URL", "http://drift-mcp.dataflow.svc.cluster.local/mcp")
AGENT_IDENTITY = os.environ.get("DEMO_AGENT_IDENTITY", "feature-agent")
FEATURE_SCOPE = os.environ.get("DEMO_FEATURE_SCOPE", "financial-distress:read")
DRIFT_SCOPE = os.environ.get("DEMO_DRIFT_SCOPE", "financial-distress:drift")
TICKER = os.environ.get("DEMO_USER_ID", "VNM")
CHUNK_ID = os.environ.get("DEMO_CHUNK_ID", "c1")
SCENARIO = {
    "name": "quarterly-debt-shift",
    "seed": 7,
    "start_quarter": 2,
    "affected_fraction": 1.0,
    "feature_shifts": {"total_liabilities": {"mode": "multiplicative", "magnitude": 1.15}},
    "target_metric": "debt_to_asset",
    "observed_stat": "mean",
    "expected_direction": "increase",
    "threshold": 0.01,
}
agent = {"identity": AGENT_IDENTITY, "scopes": [FEATURE_SCOPE, DRIFT_SCOPE], "tools": ["lookup_feature_context", "build_realtime_drift_report"]}
agent

{'identity': 'feature-agent', 'scopes': ['financial-distress:read', 'financial-distress:drift'], 'tools': ['lookup_feature_context', 'build_realtime_drift_report']}

In [2]:
async def call_tool(url: str, name: str, request: dict) -> dict:
    async with streamable_http_client(url) as (read, write, _) :
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(name, {"request": request})
            return result.structuredContent or {"content": [item.model_dump() for item in result.content]}

async def run_agent() -> dict:
    feature_request = {
        "agent_identity": AGENT_IDENTITY,
        "scope": FEATURE_SCOPE,
        "user_id": TICKER,
        "feature_names": ["company_risk_features:z_score"],
        "chunk_id": CHUNK_ID,
    }
    drift_request = {
        "agent_identity": "drift-agent",
        "scope": DRIFT_SCOPE,
        "rows": [{"ticker": TICKER, "total_assets": 100.0, "total_liabilities": 50.0, "close": 10.0}],
        "scenario": SCENARIO,
    }
    feature_result, drift_result = await asyncio.gather(
        call_tool(FEATURE_MCP_URL, "lookup_feature_context", feature_request),
        call_tool(DRIFT_MCP_URL, "build_realtime_drift_report", drift_request),
    )
    return {"agent": agent, "feature_result": feature_result, "drift_result": drift_result}

agent_result = asyncio.run(run_agent())
print(json.dumps(agent_result, indent=2, default=str))

{
  "agent": {
    "identity": "feature-agent",
    "scopes": [
      "financial-distress:read",
      "financial-distress:drift"
    ],
    "tools": [
      "lookup_feature_context",
      "build_realtime_drift_report"
    ]
  },
  "feature_result": {
    "ok": true,
    "data": {
      "features": {
        "z_score": null
      },
      "rag": {
        "chunk_id": "phase3-chunk",
        "chunk_text": "Audited Phase 03 evidence chunk.",
        "source_uri": "https://example.com/phase3",
        "company": "VNM",
        "report_date": "2026-08-10",
        "access_class": "public"
      }
    },
    "error": null
  },
  "drift_result": {
    "ok": true,
    "data": {
      "report": {
        "scenario": "quarterly-debt-shift",
        "seed": 7,
        "target_metric": "debt_to_asset",
        "observed_stat": "mean",
        "before": {
          "mean": 0.5,
          "std": 0.0,
          "p50": 0.5,
          "p95": 0.5,
          "count": 1
        },
        "after": {
   

## Interpretation

The agent uses two governed tools rather than reaching the feature store, RAG database, or drift implementation directly. The returned feature/RAG metadata and drift report are the inputs a coordinator uses to construct a cited response.